# Open the FLARE sample
Developed by BRIDGE Research and Policy Foundation and ARTH GEO Intelligence.
Install `requirements.txt`, use the six included NetCDF scenes under `data/NETCDF/`, and run each cell.
Set `FLARE_DATA_DIR` or edit DATA below to use an existing directory.
The public sample contains six scenes for July 16, 2026, 18:01–18:26 UTC inclusive.
Data and examples: CC BY-NC-ND 4.0. See the repository LICENSE.


In [1]:
from pathlib import Path
import os, sys
REPO = Path.cwd() if (Path.cwd() / 'examples').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / 'examples'))
from flare_sample import data_root, scenes, open_scene, false_rgb, nearest_pixel, interactive_map
DATA = data_root(os.environ.get('FLARE_DATA_DIR', str(REPO / 'data')))
FORMAT = 'netcdf'  # Public sample is included; use 'zarr' for an approved full download.
TAGS = scenes(DATA, FORMAT)
OUTPUT = REPO / 'outputs'
OUTPUT.mkdir(exist_ok=True)
print(f'{len(TAGS)} scenes; {TAGS[0]} through {TAGS[-1]}. All times UTC.')


6 scenes; 20260716T1801 through 20260716T1826. All times UTC.


In [2]:
ds = open_scene(DATA, TAGS[0], FORMAT)
display(ds)
for key, value in ds.attrs.items():
    print(key, ':', value)


<xarray.Dataset> Size: 36MB
Dimensions:                (y: 545, x: 1425)
Coordinates:
    lat                    (y, x) float32 3MB ...
    lon                    (y, x) float32 3MB ...
    time                   datetime64[ns] 8B ...
Dimensions without coordinates: y, x
Data variables:
    I01                    (y, x) float32 3MB ...
    I02                    (y, x) float32 3MB ...
    I03                    (y, x) float32 3MB ...
    I04                    (y, x) float32 3MB ...
    I05                    (y, x) float32 3MB ...
    BT04                   (y, x) float32 3MB ...
    BT05                   (y, x) float32 3MB ...
    FireScore              (y, x) float64 6MB ...
    ObservationConfidence  (y, x) uint8 777kB ...
    AtmosphereState        (y, x) uint8 777kB ...
Attributes:
    band_note:        I01-I05 and BT04/BT05 are reconstructed/inferred values...
    complete:         1
    developed_by:     ["BRIDGE Research and Policy Foundation", "ARTH GEO Int...
    layer_count:      12
    license:          CC BY-NC-ND 4.0
    license_url:      https://creativecommons.org/licenses/by-nc-nd/4.0/
    missing_values:   Floating NaN means unavailable. Categorical zero is val...
    product:          FLARE
    product_version:  7.14
    scene_time_utc:   2026-07-16T18:01:00Z
    time_reference:   UTC scan start
    title:            FLARE v7.14 Oregon sample

band_note : I01-I05 and BT04/BT05 are reconstructed/inferred values from the source scene, not direct VIIRS observations. Radiance unit strings are not supplied by source metadata.
complete : 1
developed_by : ["BRIDGE Research and Policy Foundation", "ARTH GEO Intelligence"]
layer_count : 12
license : CC BY-NC-ND 4.0
license_url : https://creativecommons.org/licenses/by-nc-nd/4.0/
missing_values : Floating NaN means unavailable. Categorical zero is valid and is not a fill value.
product : FLARE
product_version : 7.14
scene_time_utc : 2026-07-16T18:01:00Z
time_reference : UTC scan start
title : FLARE v7.14 Oregon sample


## Inspect the actual grid and variables
Latitude and longitude are two-dimensional and share the data grid.
All products in this sample have already been aligned to the v7.14 native ROI.
Do not apply the older v5.12 20-pixel crop offset to these files.
Valid score zero and categorical zero are retained. Floating NaN means unavailable.


In [3]:
import numpy as np
for name in ds.data_vars:
    v = ds[name]
    print(name, v.shape, v.dtype, v.attrs.get('units', 'units not supplied'))
assert ds.lat.shape == ds.lon.shape == ds.FireScore.shape == (545, 1425)
print('Latitude:', float(ds.lat.min()), float(ds.lat.max()))
print('Longitude:', float(ds.lon.min()), float(ds.lon.max()))
print('UTC scene:', ds.time.values)
print('Valid score range:', float(ds.FireScore.min()), float(ds.FireScore.max()))


I01 (545, 1425) float32 units not supplied
I02 (545, 1425) float32 units not supplied
I03 (545, 1425) float32 units not supplied
I04 (545, 1425) float32 units not supplied
I05 (545, 1425) float32 units not supplied
BT04 (545, 1425) float32 K
BT05 (545, 1425) float32 K
FireScore (545, 1425) float64 1
ObservationConfidence (545, 1425) uint8 1
AtmosphereState (545, 1425) uint8 1
Latitude: 43.45262145996094 45.7343635559082
Longitude: -124.19792938232422 -118.4687271118164
UTC scene: 2026-07-16T18:01:00.000000000
Valid score range: 0.0 100.0


## Compare scientific formats (optional)
If both formats were downloaded, check every variable in the selected scene.
Otherwise this cell explains which download is missing.


In [4]:
if (DATA/'Zarr').exists() and (DATA/'NETCDF').exists():
    with open_scene(DATA,TAGS[0],'zarr') as a, open_scene(DATA,TAGS[0],'netcdf') as b:
        assert set(a.variables) == set(b.variables)
        for name in a.variables:
            np.testing.assert_array_equal(a[name].values, b[name].values)
    print('All variables and coordinates match exactly.')
else:
    print('Download both Zarr and NetCDF to run the cross-format comparison.')
ds.close()


Download both Zarr and NetCDF to run the cross-format comparison.
